In [75]:
from ET_pylib.setup import *
setup_notebook()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Kernekraftstød: numerisk analyse

Tabeller for basisforløbet mod kernekraftforløbet i GrønREFORM, hvor ny kernekraft i DK1 og DK2 erstatter havvind. Velfærdsåret er 2040.

Notebooken har to dele: **elmarkedet** (afsnit 3–12) og **fjernvarmemarkedet** (afsnit 13–18). Stødet vælges i afsnit 1: kernekraft som kondensværk eller som udtagsværk, der også leverer fjernvarme.

*`1. opsætning`*

*vælg geografi, år og kørsel*

In [76]:
# 1. opsætning
GEO   = "disaggEA"
print("Kernekraft-id'er:", ", ".join(f"{i} (fra {y})" for i, y in use_geography(GEO).items()))

yr    = 2040
ids   = NUCLEAR_IDS_BY_GEO[GEO]
RUN   = "disaggEA_2040_kf25om"           # 2040 med brintfiks
SHOCK = "shock_nuc_2040.pkl"                # kondens (CD)
# SHOCK = "shock_nuc_chp_2040.pkl"          # kraftvarme (EX), fire varmeområder
# RUN   = "disaggEA_2025_2050"           # DREAMs 2025-2050 uden brintfiks

Kernekraft-id'er: idNUC_DK1 (fra 2035), idNUC_DK2 (fra 2035), idNUC_DK1_C (fra 2035), idNUC_DK1_D (fra 2035), idNUC_DK2_C (fra 2035), idNUC_DK2_D (fra 2035)


*`2. data`*

*indlæs basis- og stødforløb og vis kapacitetsbyttet*

In [77]:
# 2. data
baseline_output, shock_output = load_run(RUN, SHOCK)
nuc, cut = nuc_wind_swap(baseline_output, shock_output, yr=yr, nuc_ids=ids)

Kapacitetsbytte — 2040 (GW)
                    basis  stød      Δ
──────────────────────────────────────
Kernekraft DK1       0,00  0,65  +0,65
Kernekraft DK2       0,00  0,35  +0,35
Havvind DK1          4,50  3,22  -1,28
Havvind DK2          1,79  1,25  -0,55
──────────────────────────────────────
Kernekraft i alt                 +1,00
Havvind i alt                    -1,82
──────────────────────────────────────
Havvind fjernet pr. GW kernekraft: 1,82



*byg timebalancerne pr. zone og for Danmark samlet*

In [78]:
ctx_el     = build_el_balance_data(baseline_output, shock_output, years=yr)
ctx_el_agg = build_el_balance_data_country(baseline_output, shock_output, years=yr)
shares     = build_supply_shares(ctx_el, aggregates=c_aggs_countries)

## Elmarkedet

*`3. priser og knaphed`*

*efterspørgselsvægtede prismomenter og andelen af nulpris- og mangeltimer*

In [79]:
# 3. priser og knaphed
print_price_summary(ctx_el, eas=("DK1", "DK2"), countries=("DK",), years=yr)

Priser og knaphed — 2040
                                Pris (kr./MWh)             Fordeling        Timer (pct.)
                          gns.  min.    maks.  std.afv.  skævhed  kurtose  nulpris  mangel
──────────────────────────────────────────────────────────────────────────────────────────
Danmark (Vest), basis    383,9  40,4  1.179,9     277,2     1,19     0,48      0,0    13,7
Danmark (Vest), stød     382,8  40,4  1.165,5     271,6     1,18     0,47      0,0    13,7
──────────────────────────────────────────────────────────────────────────────────────────
Danmark (Øst), basis     381,0  35,4  1.184,5     271,3     1,21     0,54      0,0    15,9
Danmark (Øst), stød      378,3  35,4  1.157,3     264,7     1,20     0,55      0,0    13,7
──────────────────────────────────────────────────────────────────────────────────────────
Danmark, basis           382,9  38,7  1.169,1     274,9     1,20     0,50      0,0    13,7
Danmark, stød            381,3  38,7  1.146,2     269,0     1,19   

*`4. handel`*

*eksport, import og nettoeksport pr. handelspartner for DK, NO og SE*

In [80]:
# 4. handel
print_trade_summary(ctx_el, countries=("DK"), years=yr)

Handel — Danmark, 2040
                           Eksport (TWh)  Import (TWh)      Netto (TWh)        Netto (mio. kr.)
                            basis    stød  basis   stød  basis   stød      Δ   basis    stød     Δ
──────────────────────────────────────────────────────────────────────────────────────────────────
Tyskland og Luxembourg      16,50   16,44   4,49   3,94  12,02  12,50  +0,48   3.362   4.112  +750
Storbritannien og Irland     3,75    3,82   4,00   4,14  -0,25  -0,32  -0,07    -397    -306   +91
Nederlandene                 1,07    1,10   1,50   1,59  -0,43  -0,49  -0,06     -82     -18   +64
Norge                        1,53    1,43   8,79   8,88  -7,26  -7,45  -0,19  -1.499  -1.535   -37
Sverige                      3,02    2,97   7,64   7,66  -4,61  -4,69  -0,08  -2.363  -2.332   +31
──────────────────────────────────────────────────────────────────────────────────────────────────
I alt                       25,87   25,76  26,42  26,21  -0,53  -0,45  +0,08    -979     

*bytteforhold pr. handelspartner: eksportpris over importpris*

In [81]:
print_terms_of_trade_partner(ctx_el, country="DK", years=yr)

Bytteforhold pr. handelspartner — Danmark, 2040
                           Eksport (TWh)  Import (TWh)  Eksportpris   Importpris          Bytteforhold
                            basis    stød  basis   stød  basis  stød  basis  stød  basis   stød       Δ  Δ pct.
───────────────────────────────────────────────────────────────────────────────────────────────────────────────
Tyskland og Luxembourg      16,50   16,44   4,49   3,94    325   340    446   374  0,729  0,909  +0,179   +24,6
Storbritannien og Irland     3,75    3,82   4,00   4,14    285   297    367   348  0,778  0,853  +0,076    +9,7
Nederlandene                 1,07    1,10   1,50   1,59    413   455    350   326  1,181  1,395  +0,213   +18,1
Norge                        1,53    1,43   8,79   8,88    124   125    192   193  0,646  0,646  -0,000    -0,0
Sverige                      3,02    2,97   7,64   7,66    119   120    356   351  0,334  0,341  +0,007    +2,0
─────────────────────────────────────────────────────────────────

*`5. fangstpris og værdifaktor`*

*produktionsvægtet fangstpris pr. anlægstype i Danmark*

In [82]:
# 5. fangstpris og værdifaktor
print_capture(capture_summary(ctx_el, country="DK", years=yr), year=yr)

Fangstpris og værdifaktor, el — Danmark, 2040
                                         Produktion (TWh)     Fangstpris (kr./MWh)   Værdifaktor
                                        basis   stød      Δ    basis     stød      Δ            Δ
─────────────────────────────────────────────────────────────────────────────────────────────────
Havvind                                 41,02  33,04  -7,98    298,8    299,2   +0,4        +0,00
Solceller                               19,01  19,03  +0,02    189,1    187,6   -1,5        -0,00
Landvind                                13,96  13,98  +0,02    275,9    279,0   +3,1        +0,01
Modtryksværk (kraftvarme)                1,61   1,60  -0,00    557,7    550,7   -7,0        -0,01
Ellagring                                1,51   1,51  -0,00    655,7    645,7  -10,0        -0,02
Modtryksværk med bypass (kraftvarme)     0,44   0,43  -0,01    818,5    804,7  -13,8        -0,03
Udtagsværk (kraftvarme)                  0,40   0,39  -0,01    884,4    8

*`6. kapacitetsfaktorer`*

*realiseret produktion over installeret kapacitet gange 8760 timer*

In [83]:
# 6. kapacitetsfaktorer
print_capacity_factor(capacity_factor_summary(ctx_el, country="DK", years=yr), year=yr)

Kapacitetsfaktor, el — Danmark, 2040
                                         Produktion (TWh)   Kapacitetsfaktor (pct., ECap)
                                        basis   stød      Δ      basis      stød         Δ
──────────────────────────────────────────────────────────────────────────────────────────
Havvind                                 41,02  33,04  -7,98       50,4      50,5      +0,1
Solceller                               19,01  19,03  +0,02       12,1      12,1      +0,0
Landvind                                13,96  13,98  +0,02       27,7      27,8      +0,0
Modtryksværk (kraftvarme)                1,61   1,60  -0,00       26,9      26,8      -0,1
Ellagring                                1,51   1,51  -0,00        7,6       7,6      -0,0
Modtryksværk med bypass (kraftvarme)     0,44   0,43  -0,01       16,1      15,8      -0,3
Udtagsværk (kraftvarme)                  0,40   0,39  -0,01       13,4      13,1      -0,3
Eksogen elproduktion                     0,06   0,06  

c:\Users\LinusLindquist\e-tank\E-Tank - Data\PROJEKTER\04 KERNEKRAFT\Grøn Reform beregninger og følgegruppe\GrønREFORM_2.0\KF26_wrapper\ET_pylib\market.py:331: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.26]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mult[is_ex] = (1.0 + m.loc[is_ex, "Cv"] / m.loc[is_ex, "Cm"]).to_numpy()


*`7. forbrugeroverskud`*

*ΔCS pr. zone som trapez på DC-IIA-efterspørgslen, $\Delta CS = -\sum_h \left[D_{0h} + \tfrac{1}{2}(D_{1h} - D_{0h})\right]\Delta p_h$*

In [84]:
# 7. forbrugeroverskud
print_cs_table(baseline_output, shock_output, year=yr)

Forbrugeroverskud, el — 2040 (mio. kr.)
                                 heraf
                         ΔCS    pris  mængde
────────────────────────────────────────────
Danmark (Vest)         +44,2   +44,1    +0,1
Danmark (Øst)          +58,3   +58,3    +0,1
Danmark (Bornholm)      +0,0    +0,0    +0,0
Danmark i alt         +102,5  +102,4    +0,1
────────────────────────────────────────────
Norge (syd)            -62,4   -62,4    +0,0
Norge (nord)            -7,2    -7,2    +0,0
Norge (midt)           -12,9   -12,9    +0,0
Norge i alt            -82,6   -82,6    +0,0
────────────────────────────────────────────
Sverige (nord)          -3,7    -3,7    +0,0
Sverige (midtnord)      -6,7    -6,7    +0,0
Sverige (midtsyd)      -28,4   -28,4    +0,0
Sverige (syd)          +14,2   +14,2    +0,0
Sverige i alt          -24,5   -24,6    +0,0
────────────────────────────────────────────
I alt                   -4,6    -4,7    +0,1
────────────────────────────────────────────



*`8. producentoverskud`*

*UP gange kapacitet pr. anlægstype, opdelt i pris-, mængde- og resteffekt*

In [85]:
# 8. producentoverskud
areas = c_aggs["DK"]
maps = dict(id2Pt=baseline_output["id2Pt"].vals, id2EA=shock_output["id2EA"].vals)
dec_maps = dict(**maps,
                id2HA=shock_output["id2HA"].vals,
                id2cIdxE=shock_output["id2cIdxE"].vals,
                id2cIdxH=shock_output["id2cIdxH"].vals)

bt = producer_profit(baseline_output, shock_output, areas=areas, year=yr,
                     split_nuclear=True, **maps)
dec = profit_decomposition(baseline_output, shock_output, areas=areas, year=yr,
                           split_nuclear=True, **dec_maps)
print_producer_profit(bt, year=yr, areas=areas, decomp=dec)

Producentoverskud — Danmark, 2040 (mio. kr.)
                                            Producentoverskud                Δ, heraf
                                           basis      stød         Δ   pris  mængde      rest
─────────────────────────────────────────────────────────────────────────────────────────────
Havvind                                  6.962,9   5.630,8  -1.332,2  +91,3   -11,6  -1.411,9
Solceller                                3.136,2   3.100,6     -35,6  -36,7    +0,1      +1,1
Ellagring                                  510,7     487,6     -23,1  -23,7    +0,9      -0,3
Modtryksværk (kraftvarme)                  205,8     197,4      -8,5   -9,8    +0,2      +1,1
Udtagsværk (kraftvarme)                    126,3     118,8      -7,5   -7,5    +0,1      -0,0
Modtryksværk med bypass (kraftvarme)      -182,4    -188,0      -5,5   -6,7    +0,1      +1,1
Eksogen elproduktion                        11,8      11,0      -0,8   -0,9    +0,0      +0,1
Varmelagring           

*`9. produktion`*

*elproduktion pr. anlægstype i Danmark*

In [86]:
# 9. produktion
by_qty = producer_generation(baseline_output, shock_output, areas=areas, year=yr,
                             split_nuclear=True, **maps)
print_producer_qty(by_qty, year=yr, areas=areas)

Elproduktion — Danmark, 2040 (TWh)
                                        basis   stød      Δ
───────────────────────────────────────────────────────────
Havvind                                 41,02  33,04  -7,98
Udtagsværk (kraftvarme)                  0,40   0,39  -0,01
Modtryksværk med bypass (kraftvarme)     0,44   0,43  -0,01
Modtryksværk (kraftvarme)                1,61   1,60  -0,00
Eksogen elproduktion                     0,06   0,06  -0,00
Kondensværk                              0,00   0,00  -0,00
Vandkraft (flodkraft)                    0,02   0,02  +0,00
Ellagring                                4,20   4,21  +0,01
Landvind                                13,96  13,98  +0,02
Solceller                               26,03  26,06  +0,03
Kernekraft                               0,00   8,01  +8,01
───────────────────────────────────────────────────────────
I alt                                   87,73  87,79  +0,06



*`10. flaskehalsindtægt`*

*flaskehalsindtægt til de danske TSO'er; hver linjes rente deles ligeligt mellem de to ender*

In [87]:
# 10. flaskehalsindtægt
print_ci(baseline_output, shock_output, year=yr, country="DK")

Flaskehalsindtægt til TSO — Danmark, 2040 (mio. kr.)
                        basis     stød      Δ
─────────────────────────────────────────────
Danmark (Vest)        2.060,1  2.014,9  -45,2
Danmark (Øst)           402,1    386,6  -15,5
Danmark (Bornholm)      185,6    182,1   -3,5
─────────────────────────────────────────────
I alt                 2.647,8  2.583,6  -64,2



*`11. velfærd`*

*værdien af kernekraftens fleksibilitet: værkerne tvinges op på minimumslast, og den ekstra produktion værdisættes til modellens priser*
$$
    \Delta W = \sum_{h\in\mathcal H} (p_h-MC-s) \cdot \Delta q_h
$$

In [88]:
# 11. velfærd
print_flex(shock_output, year=yr, u_mins=(0.3))      # must-run og 30 pct. minimumslast

Fleksibilitetsværdi, kernekraft — 2040, minimumslast 30 pct.
                        TWh            kr./MWh     mio. kr.
                  produktion  ekstra  pris, ekstra        ΔW
────────────────────────────────────────────────────────────
Kernekraft DK1          5,20    0,10          62,3      -7,6
Kernekraft DK2          2,81    0,05          57,0      -4,2
────────────────────────────────────────────────────────────
I alt                   8,01    0,15          60,5     -11,8
────────────────────────────────────────────────────────────



*samlet velfærdseffekt for Danmark ved tre rentestrukturer*

In [89]:
blocks = welfare_block_by_rate(
    baseline_output, shock_output,
    {"FM": "fm", "4 pct.": 0.04, "6 pct.": 0.06},
    year=yr, regions=("DK",), markets=("E",),
)
print_welfare_blocks(blocks, regions="DK")

Velfærdseffekt — Danmark, 2040 (mio. kr./år)
                                       FM     4 pct.     6 pct.
───────────────────────────────────────────────────────────────
Δ Forbrugeroverskud               +102,51    +102,51    +102,51
Δ Producentoverskud               +113,49    +113,49    +113,49
Δ Flaskehalsindtægter              -64,20     -64,20     -64,20
Annuitet, netto                   -654,80    -885,98  -1.677,82
Systemomkostning, VE            +1.150,43  +1.150,43  +1.150,43
Systemomkostning, kernekraft      -245,03    -245,03    -245,03
───────────────────────────────────────────────────────────────
Netto velfærdseffekt              +402,40    +171,22    -620,62
───────────────────────────────────────────────────────────────
Annuitet, kernekraft (ny)       +2.504,38  +2.853,22  +4.149,17
Annuitet, havvind (nedlagt)     -1.849,58  -1.967,24  -2.471,34
───────────────────────────────────────────────────────────────
Annuitet: + = omkostning, - = sparet.
Systemomkostning, VE:

*`12. LCOE`*

*LCOE af GR's egne drifts- og produktionstal plus annuitet af investeringen*

In [90]:
# 12. LCOE
print_lcoe(shock_output, year=yr, r=0.04)

LCOE — Danmark, 2040, rente: 4,0 pct. (øre/kWh)
               kap.faktor (pct.)  annuitet   FOM    MC  LCOE
────────────────────────────────────────────────────────────
Kernekraft                  91,5      35,6  12,5  10,7  58,8
Havvind                     50,5      24,4   9,0   4,0  37,3
Landvind                    27,8      23,1   7,8   2,1  33,0
Solceller                   12,2      20,7   7,9   0,0  28,6
────────────────────────────────────────────────────────────
VE (vægtet)                 22,2      22,8   8,3   2,2  33,4
────────────────────────────────────────────────────────────



c:\Users\LinusLindquist\e-tank\E-Tank - Data\PROJEKTER\04 KERNEKRAFT\Grøn Reform beregninger og følgegruppe\GrønREFORM_2.0\KF26_wrapper\ET_pylib\costs.py:320: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  el_fac = (1 + sym_year(db, "Cv", year).div(sym_year(db, "Cm", year))).reindex(ids).fillna(1.0)


---

## Fjernvarmemarkedet

Fra her gælder figurer og tabeller fjernvarmen i de fire danske varmeområder (central og decentral i Vest- og Østdanmark). Handel og flaskehalsindtægt findes ikke på fjernvarmemarkedet.

*`13. data`*

*byg fjernvarmebalancerne pr. varmeområde*

In [91]:
# 13. data
ctx_dh = build_el_balance_data(baseline_output, shock_output, years=yr, market="H")

*`14. priser og knaphed`*

*fjernvarmepriser og knaphed pr. varmeområde og for Danmark*

In [92]:
# 14. priser og knaphed
print_price_summary(ctx_dh, eas=HA_AGGS["DK"], countries=("DK",), years=yr)

Fjernvarmepriser og knaphed — 2040
                                       Pris (kr./MWh)            Fordeling        Timer (pct.)
                                  gns.  min.  maks.  std.afv.  skævhed  kurtose  nulpris  mangel
────────────────────────────────────────────────────────────────────────────────────────────────
Vestdanmark, central, basis      216,5  62,6  372,4      79,3    -0,04    -0,79      0,0     0,0
Vestdanmark, central, stød       215,9  63,1  373,6      78,7    -0,03    -0,77      0,0     0,0
────────────────────────────────────────────────────────────────────────────────────────────────
Vestdanmark, decentral, basis    329,7  69,6  557,5     116,5    -0,86    -0,00      0,0     0,0
Vestdanmark, decentral, stød     330,4  70,1  560,4     117,0    -0,85    -0,01      0,0     0,0
────────────────────────────────────────────────────────────────────────────────────────────────
Østdanmark, central, basis       318,3  61,6  811,0     132,6     1,29     4,86      0,0     1

*`15. fangstpris og kapacitetsfaktor`*

*fangstpris og kapacitetsfaktor for fjernvarme pr. anlægstype i Danmark*

In [93]:
# 15. fangstpris og kapacitetsfaktor
print_capture(capture_summary(ctx_dh, country="DK", years=yr), year=yr, market="H")
print_capacity_factor(capacity_factor_summary(ctx_dh, country="DK", years=yr), year=yr,
                      market="H")

Fangstpris og værdifaktor, fjernvarme — Danmark, 2040
                                         Produktion (TWh)   Fangstpris (kr./MWh)  Værdifaktor
                                        basis   stød      Δ  basis   stød       Δ            Δ
──────────────────────────────────────────────────────────────────────────────────────────────
Elkedler og varmepumper                 28,49  28,52  +0,03  267,5  267,4    -0,1        -0,00
Modtryksværk (kraftvarme)                7,86   7,84  -0,02  334,7  334,7    +0,1        +0,00
Varmelagring                             5,44   5,44  -0,00  241,0  240,8    -0,3        -0,00
Varmekedel                               4,50   4,53  +0,03  349,3  350,2    +0,9        +0,00
Eksogen fjernvarmeproduktion             2,06   2,06  +0,00  244,5  244,6    +0,1        +0,00
Modtryksværk med bypass (kraftvarme)     1,90   1,88  -0,02  358,4  358,9    +0,5        +0,00
Solvarme                                 0,87   0,87  +0,00  179,3  179,0    -0,3        -0,

*`16. forbrugeroverskud`*

*forbrugeroverskud på fjernvarme pr. varmeområde*

In [94]:
# 16. forbrugeroverskud
print_cs_table(baseline_output, shock_output, year=yr, market="H")

Forbrugeroverskud, fjernvarme — 2040 (mio. kr.)
                                  heraf
                           ΔCS  pris  mængde
────────────────────────────────────────────
Vestdanmark, central      +8,3  +8,3    +0,0
Vestdanmark, decentral    -7,2  -7,2    +0,0
Østdanmark, central       -1,9  -1,9    +0,0
Østdanmark, decentral     +1,2  +1,2    +0,0
Danmark i alt             +0,3  +0,3    +0,0
────────────────────────────────────────────
I alt                     +0,3  +0,3    +0,0
────────────────────────────────────────────



*`17. produktion`*

*fjernvarmeproduktion pr. anlægstype i Danmark*

In [95]:
# 17. produktion
areas_dh = HA_AGGS["DK"]
by_qty_dh = producer_generation(baseline_output, shock_output, areas=areas_dh, year=yr,
                                split_nuclear=True, market="H", **maps)
print_producer_qty(by_qty_dh, year=yr, areas=areas_dh, market="H")

Fjernvarmeproduktion — Danmark, 2040 (TWh)
                                        basis   stød      Δ
───────────────────────────────────────────────────────────
Modtryksværk med bypass (kraftvarme)     1,90   1,88  -0,02
Modtryksværk (kraftvarme)                7,86   7,84  -0,02
Udtagsværk (kraftvarme)                  0,67   0,65  -0,02
Varmelagring                             5,44   5,44  -0,00
Power-to-X med varme                     0,44   0,44  -0,00
Solvarme                                 0,87   0,87  +0,00
Eksogen fjernvarmeproduktion             2,06   2,06  +0,00
Elkedler og varmepumper                 28,49  28,52  +0,03
Varmekedel                               4,50   4,53  +0,03
───────────────────────────────────────────────────────────
I alt                                   52,22  52,22  -0,00



*`18. velfærd`*

*velfærd med både el- og fjernvarmemarkedet (ΔCS inkl. fjernvarme)*

In [96]:
# 18. velfærd
blocks_eh = welfare_block_by_rate(
    baseline_output, shock_output,
    {"FM": "fm", "4 pct.": 0.04, "6 pct.": 0.06},
    year=yr, regions=("DK",), markets=("E", "H"),
)
print_welfare_blocks(blocks_eh, regions="DK")

Velfærdseffekt — Danmark, 2040 (mio. kr./år)
                                       FM     4 pct.     6 pct.
───────────────────────────────────────────────────────────────
Δ Forbrugeroverskud               +102,85    +102,85    +102,85
Δ Producentoverskud               +113,49    +113,49    +113,49
Δ Flaskehalsindtægter              -64,20     -64,20     -64,20
Annuitet, netto                   -654,80    -885,98  -1.677,82
Systemomkostning, VE            +1.150,43  +1.150,43  +1.150,43
Systemomkostning, kernekraft      -245,03    -245,03    -245,03
───────────────────────────────────────────────────────────────
Netto velfærdseffekt              +402,74    +171,56    -620,28
───────────────────────────────────────────────────────────────
Annuitet, kernekraft (ny)       +2.504,38  +2.853,22  +4.149,17
Annuitet, havvind (nedlagt)     -1.849,58  -1.967,24  -2.471,34
───────────────────────────────────────────────────────────────
Annuitet: + = omkostning, - = sparet.
Systemomkostning, VE: